In [1]:
!pip install fasterrisk --no-deps

In [2]:
##FasterRisk on UCI Heart Disease
from fasterrisk.fasterrisk import RiskScoreOptimizer, RiskScoreClassifier
from fasterrisk.utils import download_file_from_google_drive
import os, time
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
## 1. Sanity check on the authors' adult dataset
Path("data").mkdir(exist_ok=True)
train_path, test_path = "data/adult_train.csv", "data/adult_test.csv"
if not os.path.isfile(train_path):
    download_file_from_google_drive('1nuWn0QVG8tk3AN4I4f3abWLcFEP3WPec', train_path)
if not os.path.isfile(test_path):
    download_file_from_google_drive('1TyBO02LiGfHbatPWU4nzc8AndtIF-7WH', test_path)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
X_train, y_train = train_df.iloc[:, 1:].values, train_df.iloc[:, 0].values
X_test, y_test = test_df.iloc[:, 1:].values, test_df.iloc[:, 0].values

print(X_train.shape, X_test.shape)
print("Label values:", np.unique(y_train))
print("Example feature values:", np.unique(X_train))

(26049, 36) (6512, 36)
Label values: [-1  1]
Example feature values: [0 1]


In [4]:
m = RiskScoreOptimizer(X=X_train, y=y_train, k=5, parent_size=10)
start = time.time()
m.optimize()
print(f"Optimization took {time.time() - start:.1f} seconds")

Optimization took 31.9 seconds


In [5]:
multipliers, intercepts, coefs = m.get_models()
print(f"Number of models in the pool: {len(multipliers)}")

clf = RiskScoreClassifier(multipliers[0], intercepts[0], coefs[0], X_train=X_train)
clf.reset_featureNames(list(train_df.columns[1:]))
clf.print_model_card()

test_acc, test_auc = clf.get_acc_and_auc(X_test, y_test)
print(f"Test AUC: {test_auc:.3f}")

Number of models in the pool: 42
The Risk Score is:
1.            Age_22_to_29     -2 point(s) |   ...
2.               HSDiploma     -2 point(s) | + ...
3.                    NoHS     -4 point(s) | + ...
4.                 Married      4 point(s) | + ...
5.         AnyCapitalGains      3 point(s) | + ...
                                     SCORE | =    
SCORE |  -8.0  |  -6.0  |  -5.0  |  -4.0  |  -3.0  |  -2.0  |  -1.0  |
RISK  |   0.1% |   0.4% |   0.7% |   1.2% |   2.3% |   4.2% |   7.6% |
SCORE |   0.0  |   1.0  |   2.0  |   3.0  |   4.0  |   5.0  |   7.0  |
RISK  |  13.3% |  22.3% |  34.9% |  50.0% |  65.1% |  77.7% |  92.4% |
Test AUC: 0.856


In [6]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

heart = fetch_ucirepo(id=45)   # 45 = Heart Disease on the UCI repository
X_raw = heart.data.features
y_raw = heart.data.targets

print(X_raw.shape)
print(X_raw.head())

(303, 13)
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

    ca  thal  
0  0.0   6.0  
1  3.0   3.0  
2  2.0   7.0  
3  0.0   3.0  
4  0.0   3.0  


In [7]:
#check for missing values and look at the target
print("Missing values per column:")
print(X_raw.isna().sum())
print("\nTarget values:")
print(y_raw.value_counts())

Missing values per column:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
dtype: int64

Target values:
num
0      164
1       55
2       36
3       35
4       13
Name: count, dtype: int64


In [8]:
# clean up and create labels
df = pd.concat([X_raw, y_raw], axis=1).dropna().reset_index(drop=True)

# Original target is 0 (no disease) to 4 (severity); we make it binary.
# Coded as -1 / +1 to match FasterRisk's convention.
y = np.where(df["num"] > 0, 1, -1)
X_df = df.drop(columns="num")

print("Rows after dropping missing:", len(df))
print("Disease cases:", (y == 1).sum(), " No disease:", (y == -1).sum())

Rows after dropping missing: 297
Disease cases: 137  No disease: 160


In [9]:
 ##  Binarize features with clinical thresholds
def b(cond):
    return cond.astype(int)

d = X_df
Xb = pd.DataFrame({
    "Age>=50":               b(d.age >= 50),
    "Age>=60":               b(d.age >= 60),
    "Male":                  b(d.sex == 1),
    "ChestPain_Atypical":    b(d.cp == 2),
    "ChestPain_NonAnginal":  b(d.cp == 3),
    "ChestPain_Asymptomatic":b(d.cp == 4),
    "RestBP>=130":           b(d.trestbps >= 130),
    "RestBP>=140":           b(d.trestbps >= 140),
    "Chol>=200":             b(d.chol >= 200),
    "Chol>=240":             b(d.chol >= 240),
    "FastingSugar>120":      b(d.fbs == 1),
    "RestECG_Abnormal":      b(d.restecg > 0),
    "MaxHR<120":             b(d.thalach < 120),
    "MaxHR<140":             b(d.thalach < 140),
    "MaxHR<160":             b(d.thalach < 160),
    "ExerciseAngina":        b(d.exang == 1),
    "STDepression>=1":       b(d.oldpeak >= 1),
    "STDepression>=2":       b(d.oldpeak >= 2),
    "Slope_Flat":            b(d.slope == 2),
    "Slope_Down":            b(d.slope == 3),
    "Vessels>=1":            b(d.ca >= 1),
    "Vessels>=2":            b(d.ca >= 2),
    "Thal_Fixed":            b(d.thal == 6),
    "Thal_Reversible":       b(d.thal == 7),
})

print(Xb.shape)
print("\nFraction of patients with each feature = 1:")
print(Xb.mean().round(2))

(297, 24)

Fraction of patients with each feature = 1:
Age>=50                   0.71
Age>=60                   0.31
Male                      0.68
ChestPain_Atypical        0.16
ChestPain_NonAnginal      0.28
ChestPain_Asymptomatic    0.48
RestBP>=130               0.56
RestBP>=140               0.33
Chol>=200                 0.84
Chol>=240                 0.52
FastingSugar>120          0.14
RestECG_Abnormal          0.51
MaxHR<120                 0.11
MaxHR<140                 0.28
MaxHR<160                 0.61
ExerciseAngina            0.33
STDepression>=1           0.46
STDepression>=2           0.20
Slope_Flat                0.46
Slope_Down                0.07
Vessels>=1                0.41
Vessels>=2                0.20
Thal_Fixed                0.06
Thal_Reversible           0.39
dtype: float64


In [10]:
# first look at a heart disease risk score
X = Xb.values
feature_names = list(Xb.columns)

m = RiskScoreOptimizer(X=X, y=y, k=5, parent_size=10)
m.optimize()
multipliers, intercepts, coefs = m.get_models()

clf = RiskScoreClassifier(multipliers[0], intercepts[0], coefs[0], X_train=X)
clf.reset_featureNames(feature_names)
clf.print_model_card()

acc, auc = clf.get_acc_and_auc(X, y)
print(f"Training AUC (optimistic): {auc:.3f}")

The Risk Score is:
1. ChestPain_Asymptomatic      4 point(s) |   ...
2.        STDepression>=2      4 point(s) | + ...
3.             Slope_Flat      3 point(s) | + ...
4.             Vessels>=1      5 point(s) | + ...
5.        Thal_Reversible      4 point(s) | + ...
                                    SCORE | =    
SCORE |   0.0  |   3.0  |   4.0  |   5.0  |   7.0  |   8.0  |   9.0  |
RISK  |   2.6% |   9.4% |  14.1% |  20.5% |  38.9% |  50.0% |  61.1% |
SCORE |  11.0  |  12.0  |  13.0  |  15.0  |  16.0  |  17.0  |  20.0  |
RISK  |  79.5% |  85.9% |  90.6% |  95.9% |  97.4% |  98.3% |  99.6% |
Training AUC (optimistic): 0.919


In [11]:
## Cross-validated comparison with logistic regression
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def sparse_and_rounded_lr(Xtr, ytr, Xte, k, max_points=5):
    # Pick the k features with the largest coefficients in a full model
    full = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
    top = np.argsort(-np.abs(full.coef_[0]))[:k]
    # Refit using only those k features
    sub = LogisticRegression(max_iter=5000).fit(Xtr[:, top], ytr)
    w = sub.coef_[0]
    sparse_scores = Xte[:, top] @ w
    # Naive rounding: scale so the largest weight = max_points, then round
    w_int = np.round(w * max_points / np.max(np.abs(w)))
    rounded_scores = Xte[:, top] @ w_int
    return sparse_scores, rounded_scores

In [12]:
#  run the cross-validation experiment
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
results = {}

def record(name, value):
    results.setdefault(name, []).append(value)

for fold, (tr, te) in enumerate(skf.split(X, y)):
    Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]

    full = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
    record("LR full (24 features)", roc_auc_score(yte, full.decision_function(Xte)))

    for k in [3, 5, 7]:
        s, r = sparse_and_rounded_lr(Xtr, ytr, Xte, k)
        record(f"LR sparse k={k}", roc_auc_score(yte, s))
        record(f"LR rounded k={k}", roc_auc_score(yte, r))

        m = RiskScoreOptimizer(X=Xtr, y=ytr, k=k, parent_size=10)
        m.optimize()
        mult, b0, betas = m.get_models()
        clf = RiskScoreClassifier(mult[0], b0[0], betas[0])
        record(f"FasterRisk k={k}", roc_auc_score(yte, clf.predict_prob(Xte)))

    print(f"Fold {fold + 1} done")

summary = pd.DataFrame(
    {name: [np.mean(v), np.std(v)] for name, v in results.items()},
    index=["mean test AUC", "std"]
).T.round(3)
print(summary)

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done
                       mean test AUC    std
LR full (24 features)          0.902  0.034
LR sparse k=3                  0.839  0.034
LR rounded k=3                 0.841  0.031
FasterRisk k=3                 0.884  0.048
LR sparse k=5                  0.902  0.023
LR rounded k=5                 0.903  0.020
FasterRisk k=5                 0.904  0.032
LR sparse k=7                  0.906  0.026
LR rounded k=7                 0.903  0.029
FasterRisk k=7                 0.910  0.030


In [13]:
## Why FasterRisk wins at k=3
fr = np.array(results["FasterRisk k=3"])
lr = np.array(results["LR sparse k=3"])
print("Fold | FasterRisk | LR sparse | Difference")
for i in range(5):
    print(f"  {i+1}  |   {fr[i]:.3f}    |   {lr[i]:.3f}   |  {fr[i]-lr[i]:+.3f}")
print(f"FasterRisk wins in {(fr > lr).sum()} of 5 folds")

Fold | FasterRisk | LR sparse | Difference
  1  |   0.926    |   0.796   |  +0.129
  2  |   0.950    |   0.884   |  +0.065
  3  |   0.825    |   0.811   |  +0.014
  4  |   0.839    |   0.832   |  +0.007
  5  |   0.878    |   0.872   |  +0.006
FasterRisk wins in 5 of 5 folds


In [14]:
#  which 3 features does each method choose?
full = LogisticRegression(max_iter=5000).fit(X, y)
top3 = np.argsort(-np.abs(full.coef_[0]))[:3]
print("LR baseline picks:")
for i in top3:
    print(f"  {feature_names[i]:25s} coef={full.coef_[0][i]:+.2f}  prevalence={X[:, i].mean():.2f}")

m3 = RiskScoreOptimizer(X=X, y=y, k=3, parent_size=10)
m3.optimize()
mult, b0, betas = m3.get_models()
c3 = RiskScoreClassifier(mult[0], b0[0], betas[0], X_train=X)
c3.reset_featureNames(feature_names)
print("\nFasterRisk picks:")
c3.print_model_card()

LR baseline picks:
  Vessels>=1                coef=+1.76  prevalence=0.41
  Thal_Reversible           coef=+1.37  prevalence=0.39
  ChestPain_Asymptomatic    coef=+1.35  prevalence=0.48

FasterRisk picks:
The Risk Score is:
1. ChestPain_Asymptomatic      3 point(s) |   ...
2.             Vessels>=1      3 point(s) | + ...
3.        Thal_Reversible      3 point(s) | + ...
                                    SCORE | =    
SCORE |   0.0  |   3.0  |
RISK  |   5.5% |  33.0% |
SCORE |   6.0  |   9.0  |
RISK  |  80.5% |  97.2% |


In [15]:
#  compare feature choices fold by fold (same splits as before)
for fold, (tr, te) in enumerate(skf.split(X, y)):
    Xtr, ytr = X[tr], y[tr]

    full = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
    lr_feats = sorted(feature_names[i] for i in np.argsort(-np.abs(full.coef_[0]))[:3])

    m = RiskScoreOptimizer(X=Xtr, y=ytr, k=3, parent_size=10)
    m.optimize()
    _, _, betas = m.get_models()
    fr_feats = sorted(feature_names[i] for i in np.nonzero(betas[0])[0])

    same = "SAME" if lr_feats == fr_feats else "DIFFERENT"
    print(f"Fold {fold+1} [{same}]")
    print(f"  LR:         {lr_feats}")
    print(f"  FasterRisk: {fr_feats}\n")

Fold 1 [DIFFERENT]
  LR:         ['Male', 'STDepression>=2', 'Vessels>=1']
  FasterRisk: ['ChestPain_Asymptomatic', 'Thal_Reversible', 'Vessels>=1']

Fold 2 [DIFFERENT]
  LR:         ['Male', 'Thal_Reversible', 'Vessels>=1']
  FasterRisk: ['ChestPain_Asymptomatic', 'Thal_Reversible', 'Vessels>=1']

Fold 3 [SAME]
  LR:         ['ChestPain_Asymptomatic', 'Thal_Reversible', 'Vessels>=1']
  FasterRisk: ['ChestPain_Asymptomatic', 'Thal_Reversible', 'Vessels>=1']

Fold 4 [SAME]
  LR:         ['ChestPain_Asymptomatic', 'Thal_Reversible', 'Vessels>=1']
  FasterRisk: ['ChestPain_Asymptomatic', 'Thal_Reversible', 'Vessels>=1']

Fold 5 [SAME]
  LR:         ['ChestPain_Asymptomatic', 'Thal_Reversible', 'Vessels>=1']
  FasterRisk: ['ChestPain_Asymptomatic', 'Thal_Reversible', 'Vessels>=1']



In [16]:
##  The pool of near-optimal models
from collections import Counter

m5 = RiskScoreOptimizer(X=X, y=y, k=5, parent_size=10)
m5.optimize()
mult, b0, betas = m5.get_models()
print(f"Pool size: {len(mult)} models\n")

for i in range(min(10, len(mult))):
    c = RiskScoreClassifier(mult[i], b0[i], betas[i], X_train=X)
    _, auc = c.get_acc_and_auc(X, y)
    feats = [f"{feature_names[j]}({int(betas[i][j]):+d})" for j in np.nonzero(betas[i])[0]]
    print(f"Model {i+1:2d}: AUC={auc:.3f} | " + ", ".join(feats))

counts = Counter(feature_names[j] for i in range(len(mult)) for j in np.nonzero(betas[i])[0])
print(f"\nHow often each feature appears across all {len(mult)} models:")
for name, n in counts.most_common():
    print(f"  {name:25s} {n}")

Pool size: 19 models

Model  1: AUC=0.919 | ChestPain_Asymptomatic(+4), STDepression>=2(+4), Slope_Flat(+3), Vessels>=1(+5), Thal_Reversible(+4)
Model  2: AUC=0.917 | Male(+2), ChestPain_Asymptomatic(+3), Slope_Flat(+2), Vessels>=1(+3), Thal_Reversible(+2)
Model  3: AUC=0.912 | ChestPain_Asymptomatic(+4), ExerciseAngina(+2), STDepression>=2(+4), Vessels>=1(+5), Thal_Reversible(+4)
Model  4: AUC=0.914 | ChestPain_Asymptomatic(+4), STDepression>=1(+2), Slope_Flat(+2), Vessels>=1(+5), Thal_Reversible(+4)
Model  5: AUC=0.914 | ChestPain_Asymptomatic(+4), ExerciseAngina(+2), Slope_Flat(+3), Vessels>=1(+5), Thal_Reversible(+4)
Model  6: AUC=0.911 | ChestPain_Asymptomatic(+4), Slope_Flat(+3), Slope_Down(+3), Vessels>=1(+4), Thal_Reversible(+4)
Model  7: AUC=0.908 | ChestPain_Asymptomatic(+5), MaxHR<140(+2), STDepression>=2(+4), Vessels>=1(+5), Thal_Reversible(+5)
Model  8: AUC=0.912 | Male(+2), ChestPain_Asymptomatic(+4), STDepression>=2(+3), Vessels>=1(+4), Thal_Reversible(+3)
Model  9: AUC=

In [17]:
import sklearn, fasterrisk
print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

numpy 2.1.3 | pandas 2.2.3 | sklearn 1.6.1


# Interpretable Heart Disease Risk Scores with FasterRisk

A short exploration applying FasterRisk (Liu et al., NeurIPS 2022) to the UCI
Heart Disease (Cleveland) dataset and comparing it with logistic regression baselines.

## Data
297 patients (6 with missing values removed); 137 with heart disease (num > 0).
The 13 original features were converted into 24 binary features using clinical
thresholds (e.g., resting BP >= 130 / >= 140, cholesterol >= 200 / >= 240) and
one-hot encoding for categorical variables.

## Methods
5-fold stratified cross-validation (random_state=0), comparing:
- Full logistic regression on all 24 features
- Sparse LR: keep the top-k features by |coefficient|, then refit
- Rounded LR: sparse LR with weights scaled so the largest is 5, then rounded
- FasterRisk with k = 3, 5, 7 (parent_size=10)

## Results (mean test AUC ± std over 5 folds)
| Method | k=3 | k=5 | k=7 |
|---|---|---|---|
| Sparse LR | 0.839 ± 0.034 | 0.902 ± 0.023 | 0.906 ± 0.026 |
| Rounded LR | 0.841 ± 0.031 | 0.903 ± 0.020 | 0.903 ± 0.029 |
| FasterRisk | 0.884 ± 0.048 | 0.904 ± 0.032 | 0.910 ± 0.030 |

Full LR (24 features): 0.902 ± 0.034

## Observations
1. A 5-feature integer score matched full logistic regression (0.904 vs 0.902).
2. At k=3, FasterRisk won in all 5 folds. The gap came mainly from folds 1 and 2,
   where the baseline swapped in "Male"; FasterRisk selected the same three
   features in every fold. Naive rounding itself cost almost nothing here,
   likely because all features are binary.
3. The k=3 score assigns 3 points to each feature, so it reduces to counting
   red flags: 0, 1, 2, or 3 flags give 5.5%, 33%, 80.5%, and 97.2% risk.
4. All 19 near-optimal 5-feature models share the same core three features;
   the remaining slots rotate among exercise-test measurements (ST depression,
   ST slope, exercise angina, max heart rate), which appear interchangeable.

## Limitations
Small single dataset; simple baselines (a stronger feature-selection method,
such as an L1 regularization path, might narrow the k=3 gap); the model-pool
analysis uses training AUC. Run on Colab with fasterrisk installed via
`pip install fasterrisk --no-deps` and newer package versions
(numpy X, pandas X, scikit-learn X).